# Omnichannel ROPO Analytics Pipeline
## Statistical Validation & Business Intelligence


**Technology Stack:**
- Python
- Pandas
- NumPy
- SciPy
- PostgreSQL
- SQLAlchemy
- Power BI

---

### Objective

The objective of this notebook is to perform statistical validation on the processed ROPO dataset using 
hypothesis testing techniques and generate business insights. The validated analytical data warehouse will 
then be connected to Power BI for interactive visualization and decision-making.

# Step 1 : Import Required Libraries

In [30]:
import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency
from scipy.stats import ttest_ind

import matplotlib.pyplot as plt
import seaborn as sns

# Step 2 : Load Dataset

In [31]:
df = pd.read_csv("processed_ropo_dataset.csv")

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [32]:
# Verify

df.head()

,product_title_digital,rating_digital,transaction_id_digital,date_digital,channel_digital,region_digital,customer_id,quantity_digital,price_digital,revenue_digital,...,date_store,channel_store,region_store,quantity_store,price_store,revenue_store,Time_To_Conversion,Digital_Revenue_Contribution,ROPO_Flag,Conversion_Category
0,Reasons to Stay Alive,Two,10,2022-05-13,Mobile,Europe,17743,1,24.443538,24.443538,...,2023-12-17,Store,Asia,4,22.825231,91.300922,583,91.300922,Yes,Long-Term Converter
1,Black Dust,Five,29,2019-03-12,Online,US,41577,5,28.616701,143.083505,...,2024-12-09,Store,Asia,3,21.007354,63.022061,2099,63.022061,Yes,Long-Term Converter
2,Wall and Piece,Four,74,2020-03-29,Mobile,Asia,19856,3,45.865475,137.596426,...,2020-12-20,Store,Asia,2,16.499386,32.998771,266,32.998771,Yes,Long-Term Converter
3,Sapiens: A Brief History of Humankind,Five,119,2023-05-19,Online,US,12487,4,51.616801,206.467203,...,2024-09-01,Store,Asia,5,44.121294,220.606469,471,220.606469,Yes,Long-Term Converter
4,Throwing Rocks at the Google Bus: How Growth B...,Three,6817,2022-01-08,Online,Europe,12487,1,28.844856,28.844856,...,2024-09-01,Store,Asia,5,44.121294,220.606469,967,220.606469,Yes,Long-Term Converter


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 367 entries, 0 to 366
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   product_title_digital         367 non-null    object 
 1   rating_digital                367 non-null    object 
 2   transaction_id_digital        367 non-null    int64  
 3   date_digital                  367 non-null    object 
 4   channel_digital               367 non-null    object 
 5   region_digital                367 non-null    object 
 6   customer_id                   367 non-null    int64  
 7   quantity_digital              367 non-null    int64  
 8   price_digital                 367 non-null    float64
 9   revenue_digital               367 non-null    float64
 10  product_title_store           367 non-null    object 
 11  rating_store                  367 non-null    object 
 12  transaction_id_store          367 non-null    int64  
 13  date_

In [34]:
df.shape

(367, 23)

# Step 3 : Statistical Validation

In [35]:
# Create Contingency Table

contingency_table = pd.crosstab(
    df["channel_digital"],
    df["region_digital"]
)

contingency_table

region_digital,Asia,Europe,US
channel_digital,,,
Mobile,65,76,46
Online,57,59,64


In [36]:
# Perform Chi-Square Test

chi2_statistic, chi_square_p, chi_square_dof, expected = chi2_contingency(contingency_table)

print("Chi-Square Statistic :", round(chi2_statistic, 4))
print("P-Value :", round(chi_square_p, 4))
print("Degrees of Freedom :", chi_square_dof)

Chi-Square Statistic : 5.4793
P-Value : 0.0646
Degrees of Freedom : 2


In [37]:
# Decision

alpha = 0.05

if chi_square_p < alpha:
    print("Decision : Reject the Null Hypothesis (H₀)")
else:
    print("Decision : Accept the Null Hypothesis (H₀)")

Decision : Accept the Null Hypothesis (H₀)


## Step 3.2 : Independent T-Test

In [38]:
# Create Online and Mobile Revenue Groups

online_revenue = df.loc[
    df["channel_digital"] == "Online",
    "Digital_Revenue_Contribution"
]

mobile_revenue = df.loc[
    df["channel_digital"] == "Mobile",
    "Digital_Revenue_Contribution"
]

print("Online Customers :", len(online_revenue))
print("Mobile Customers :", len(mobile_revenue))

Online Customers : 180
Mobile Customers : 187


In [39]:
# Perform Independent T-Test

t_statistic, t_test_p = ttest_ind(
    online_revenue,
    mobile_revenue
)

print("T-Statistic :", round(t_statistic, 4))
print("P-Value :", round(t_test_p, 4))

T-Statistic : 0.2014
P-Value : 0.8405


In [40]:
# Decision

alpha = 0.05

if t_test_p < alpha:
    print("Decision : Reject the Null Hypothesis (H₀)")
else:
    print("Decision : Accept the Null Hypothesis (H₀)")

Decision : Accept the Null Hypothesis (H₀)


## Step 3.3 : Statistical Validation Summary

In [41]:
# Create Statistical Validation Summary

summary = pd.DataFrame({

    "Statistical Test": [
        "Chi-Square Test",
        "Independent T-Test"
    ],

    "Null Hypothesis": [
        "Channel and Region are Independent",
        "Mean Revenue is Equal"
    ],

    "P-Value": [
        round(chi_square_p, 4),
        round(t_test_p, 4)
    ],

    "Decision": [
        "H₀ Accepted",
        "H₀ Accepted"
    ],

    "Conclusion": [
        "No Significant Association",
        "No Significant Difference"
    ]

})

summary

,Statistical Test,Null Hypothesis,P-Value,Decision,Conclusion
0,Chi-Square Test,Channel and Region are Independent,0.0646,H₀ Accepted,No Significant Association
1,Independent T-Test,Mean Revenue is Equal,0.8405,H₀ Accepted,No Significant Difference


# Step 4 : Presentation Layer

## Step 4.1 : Connect Power BI to PostgreSQL